# H&N vs Lung – Rybrevant (IV + FASPRO) account segmentation, past 52 weeks (L2/L3)

**What changed vs the FASPRO YTD notebook**

| | FASPRO YTD notebook | **This notebook** |
|---|---|---|
| Products for the Lung universe | Rybrevant FASPRO only | **Rybrevant IV + Rybrevant FASPRO** (HCPs are switching from IV to FASPRO) |
| Window | YTD 2026 (2026-01-01 → 2026-10-22) | **Past 52 weeks** ending 2026-10-22 (2025-10-24 → 2026-10-22) |
| Chargemaster | CY-2026 annual rows | **Monthly rows** overlapping the 52 weeks |
| McKesson | FASPRO alerts only | **All Rybrevant alerts** (IV, FASPRO, Rybrevant + TKI) |
| DDD sales split weekly | FASPRO | Setting `SPLIT_PRODUCTS` (default **FASPRO**); IV can be added |

**Universes**
| Universe | Sources |
|---|---|
| **H&N** | LE.AI H&N HCPs (`Rybrevant_HNSCC_HV_CLAIMS`) + Chargemaster HNSCC encounters + McKesson H&N diagnosis |
| **Lung (Rybrevant IV + FASPRO)** | DDD, Xponent, MX claims, RX claims, McKesson EMR, Chargemaster NSCLC |

**Account segment (52 weeks):** both → **H&N + Lung**; H&N only → **H&N only**; Lung only → **Lung only**.

**Weekly DDD rule:** for each account and week with a DDD sale of a `SPLIT_PRODUCTS` product, the account is **Lung present** if:
- it is in the Lung universe from the **non-DDD** sources (52 weeks), **or**
- it bought **any Rybrevant (IV or FASPRO) in DDD in the 13 weeks before** that week.

**Levels:** L0 HCP · L1 outlet (DDD) · L2 hospital/account · L3 larger account. Everything is rolled up to **L2/L3** with OPAL (L2 when available, else L3).

Sections: 1 setup · 2 window · 3 lookups · 4 H&N universe · 5 Rybrevant Lung activity · 6 account segments (52 weeks) · 7 weekly DDD split · 8 insights + figures · 9 segment profiles · 10 save

## 1. Setup

### What this cell does: settings, helpers, and loading
- **Settings:**
  - `END_DATE` / `WINDOW_WEEKS`: 52 weeks ending 2026-10-22.
  - `RYB_NDC`: Rybrevant IV and FASPRO NDCs (data-sources file); Lazcluze excluded.
  - `LUNG_PRODUCTS`: products that make an account **Lung** (IV + FASPRO).
  - `SPLIT_PRODUCTS`: DDD products split weekly in section 7. Default: FASPRO.
  - `LOOKBACK_WEEKS = 13`: weekly rule, DDD only, weeks **before** the sale.
  - `CM_HNSCC_AS_HN`, `MCKESSON_HN_DX_AS_HN`: Chargemaster HNSCC / McKesson H&N diagnosis count as H&N (team decision).
  - `HN_ACCOUNT_SOURCE`: `"opal"` (OPAL `L2_L3`) or `"cohort"` (LE.AI's own mapped account when L2/L3).
- **Helpers:**
  - `clean()`: empty text and `"-"` become null.
  - `pick()`: finds a column even with hidden line breaks.
  - `week_of()`: Monday of the week.
  - `ryb_product_*()`: product from NDC or product name.

In [ ]:
import datetime as dt
import re
import dataiku
import dataiku.spark as dkuspark
from pyspark.sql import SparkSession, SQLContext, Window
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()
sqlContext = SQLContext(spark.sparkContext)

# ---------------- settings ----------------
DATASETS = {
    "hn_cohort":    "Rybrevant_HNSCC_HV_CLAIMS",
    "mx":           "MX_RYB_CLAIMS_copy",
    "rx":           "RYB_LINK_CLAIMS_copy",
    "dmn":          "dmn_cust_copy",
    "opal":         "OPAL_Affiliation_Levels",
    "ddd":          "RYB_DDD_Sales_copy",
    "xpon":         "RYB_Xpon_Sales_copy",
    "mckesson":     "Mckesson_GPO_DATA_copy",
    "chargemaster": "ChargeMaster_WeeklyReporting_20261012_v2_copy",
}
END_DATE = "2026-10-22"                 # "today"
WINDOW_WEEKS = 52
LOOKBACK_WEEKS = 13
RYB_NDC = {   # RYB_HN_Data_Sources.csv
    "57894050100": "RYBREVANT", "57894050101": "RYBREVANT",
    "57894051001": "RYBREVANT FASPRO", "57894051401": "RYBREVANT FASPRO",
    "57894051501": "RYBREVANT FASPRO", "57894052201": "RYBREVANT FASPRO",
}
LUNG_PRODUCTS = ["RYBREVANT", "RYBREVANT FASPRO"]     # IV + FASPRO make an account Lung
SPLIT_PRODUCTS = ["RYBREVANT FASPRO"]                 # DDD products split weekly (add "RYBREVANT" to split IV too)
CM_HNSCC_AS_HN = True
MCKESSON_HN_DX_AS_HN = True
HN_ACCOUNT_SOURCE = "opal"
RECENT_WEEKS = 3
# -------------------------------------------

end_dt = dt.datetime.strptime(END_DATE, "%Y-%m-%d").date()
START_DATE = (end_dt - dt.timedelta(days=7 * WINDOW_WEEKS - 1)).isoformat()
MISSING = ["", "-", "--", "NA", "N/A", "NULL", "NONE", "NAN"]

def load(name):
    return dkuspark.get_dataframe(sqlContext, dataiku.Dataset(name))

def clean(col):
    """trim; empty text and placeholders like "-" -> null"""
    s = F.trim(col.cast("string"))
    return F.when(~F.upper(s).isin(MISSING), s)

def pick(d, name):
    """find a column even if its real name has extra spaces, line breaks or different punctuation/case"""
    key = lambda x: re.sub(r"[^0-9a-z]", "", x.lower())
    found = {key(x): x for x in d.columns}
    if key(name) not in found:
        raise KeyError(f"column '{name}' not found. Available: {d.columns}")
    return F.col(f"`{found[key(name)]}`")

def week_of(col):
    """Monday of the week (same week definition for every source)"""
    return F.to_date(F.date_trunc("week", F.to_date(col)))

ndc_map = F.create_map(*[F.lit(x) for kv in RYB_NDC.items() for x in kv])
def ryb_product_ndc(col):
    return ndc_map[F.regexp_replace(F.trim(col.cast("string")), "[^0-9]", "")]

def ryb_product_name(col):
    n = F.upper(F.coalesce(col, F.lit("")))
    return F.when(n.contains("FASPRO"), "RYBREVANT FASPRO").when(n.contains("RYBREVANT"), "RYBREVANT")

df = {k: load(v) for k, v in DATASETS.items()}
for k, v in df.items():
    print(f"{k:13s} {DATASETS[k]:48s} rows={v.count():>12,}")

## 2. Date window

### What this cell does: show the 52-week window and each source's dates
- Window = 52 weeks (364 days) ending `END_DATE`.
- Shows each source's earliest and latest date. Sources that start after the window start have less than 52 weeks.
- Lists the Chargemaster time frames; section 5 uses the **monthly** rows that overlap the window.

In [ ]:
print(f"Window: {START_DATE} .. {END_DATE} ({WINDOW_WEEKS} weeks)\n")
for key, colname in [("ddd", "caln_dt"), ("xpon", "caln_dt"), ("mx", "service_date"), ("rx", "fill_date"),
                     ("mckesson", "diagnosis_date")]:
    r = df[key].agg(F.min(F.to_date(F.col(colname))).alias("min"), F.max(F.to_date(F.col(colname))).alias("max")).collect()[0]
    covers = r["min"] is not None and str(r["min"]) <= START_DATE
    print(f"{key:9s} {colname:15s} {r['min']} .. {r['max']}   starts before window: {'YES' if covers else 'NO'}")

cm = df["chargemaster"]
print("\nChargemaster cadences:")
cm.groupBy(pick(cm, "Cadence").alias("cadence")).count().show(truncate=False)

## 3. Lookups: NPI → J&J ID → L2/L3 account

### What this cell does: build the ID lookups used by every source
- **NPI → J&J ID** (dmn_cust): if an NPI has several `jnj_id`s, keep the latest `efftv_dt`.
- **Any J&J ID → L2/L3 account** (OPAL), first match wins:
  1. HCP (column `0`) → its `L2_L3`;
  2. outlet (column `1`) → most frequent `L2_L3` on its rows;
  3. ID is itself L2 (column `2`) → itself;
  4. ID is itself L3 (column `3`) → itself.
- **Account level:** L2 if the ID appears in column `2`, else L3.

In [ ]:
npi_jnj = (df["dmn"].select(clean(F.col("npi_id")).alias("npi"), clean(F.col("jnj_id")).alias("jnj_id"),
                            F.to_date("efftv_dt").alias("efftv_dt"))
           .filter("npi IS NOT NULL AND jnj_id IS NOT NULL")
           .withColumn("rk", F.row_number().over(Window.partitionBy("npi").orderBy(F.desc_nulls_last("efftv_dt"), "jnj_id")))
           .filter("rk = 1").drop("rk", "efftv_dt").cache())

o = df["opal"].select(*[clean(F.col(f"`{x}`")).alias(n) for x, n in
                        [("0", "hcp"), ("1", "l1"), ("2", "l2"), ("3", "l3"), ("L2_L3", "account_id")]])
by_hcp = (o.filter("hcp IS NOT NULL AND account_id IS NOT NULL")
           .select(F.col("hcp").alias("jnj_id"), "account_id", F.lit(1).alias("prio")))
by_l1 = (o.filter("l1 IS NOT NULL AND account_id IS NOT NULL").groupBy(F.col("l1").alias("jnj_id"), "account_id").count()
          .withColumn("rk", F.row_number().over(Window.partitionBy("jnj_id").orderBy(F.desc("count"), "account_id")))
          .filter("rk = 1").select("jnj_id", "account_id", F.lit(2).alias("prio")))
by_l2 = o.filter("l2 IS NOT NULL").select(F.col("l2").alias("jnj_id"), F.col("l2").alias("account_id"), F.lit(3).alias("prio"))
by_l3 = o.filter("l3 IS NOT NULL").select(F.col("l3").alias("jnj_id"), F.col("l3").alias("account_id"), F.lit(4).alias("prio"))
id_to_account = (by_hcp.unionByName(by_l1).unionByName(by_l2).unionByName(by_l3).distinct()
                 .withColumn("rk", F.row_number().over(Window.partitionBy("jnj_id").orderBy("prio", "account_id")))
                 .filter("rk = 1").select("jnj_id", "account_id").cache())
account_level = (by_l2.select("account_id", F.lit("L2").alias("level"))
                 .unionByName(by_l3.select("account_id", F.lit("L3").alias("level")))
                 .groupBy("account_id").agg(F.min("level").alias("account_level")).cache())

def to_account(d, id_col):
    return d.join(id_to_account.withColumnRenamed("jnj_id", id_col), id_col, "left")

print("NPIs in lookup:", f"{npi_jnj.count():,}", "| J&J IDs resolvable to an account:", f"{id_to_account.count():,}")

## 4. H&N universe (LE.AI)

### What this cell does: put the LE.AI H&N HCPs on their L2/L3 accounts
- `"-"` in `J&J ID` is treated as missing; the J&J ID then comes from dmn_cust via the NPI.
- Account per HCP: `HN_ACCOUNT_SOURCE` (`"opal"` = OPAL `L2_L3`).
- `hn_accounts`: one row per account with the number of H&N HCPs and their H&N treating patients (LE.AI, 2 years).

In [ ]:
h = df["hn_cohort"]
coh = (h.select(clean(pick(h, "NPI")).alias("npi"), clean(pick(h, "J&J ID")).alias("cohort_jnj_id"),
                clean(pick(h, "Mapped Account JNJ ID")).alias("cohort_account_id"),
                F.upper(clean(pick(h, "Mapped Account JNJ Level"))).alias("cohort_account_level"),
                clean(pick(h, "Mapped Account Name")).alias("cohort_account_name"),
                clean(pick(h, "Treating Patient Volume - HV (2 years)")).cast("double").alias("hn_pts_2y"))
         .join(npi_jnj.select("npi", F.col("jnj_id").alias("dmn_jnj_id")), "npi", "left")
         .withColumn("jnj_id", F.coalesce("cohort_jnj_id", "dmn_jnj_id")))
coh = to_account(coh, "jnj_id").withColumnRenamed("account_id", "opal_account_id")
coh = coh.withColumn("cohort_account", F.when(F.col("cohort_account_level").isin("L2", "L3"), F.col("cohort_account_id")))
acc_col = F.col("opal_account_id") if HN_ACCOUNT_SOURCE == "opal" else F.coalesce("cohort_account", "opal_account_id")
hcp_acc = (coh.withColumn("account_id", acc_col).filter("account_id IS NOT NULL")
              .groupBy("account_id", "npi").agg(F.max("hn_pts_2y").alias("hn_pts_2y")))
hn_accounts = (hcp_acc.groupBy("account_id")
                      .agg(F.countDistinct("npi").alias("hn_hcps"), F.sum("hn_pts_2y").alias("hn_pts_2y")).cache())
print("LE.AI HCPs placed on an account:", hcp_acc.select("npi").distinct().count(), "of", coh.select("npi").distinct().count())
print("H&N accounts:", f"{hn_accounts.count():,}")

## 5. Rybrevant (IV + FASPRO) Lung activity by source

### What this cell does: collect all Rybrevant activity and place it on an account, a date and a product
- One long table, `activity`: account, day, week, source, product, amount.
- **DDD:** `prod_nm` contains RYBREVANT (IV + FASPRO); buyer → account; amount = equivalized units.
- **Xponent:** `prod_nm` contains RYBREVANT; HCP → account; amount = TRx (> 0).
- **MX claims:** Rybrevant NDCs (IV + FASPRO); `rendering_npi` → account; amount = infusions (patient + day + NDC).
- **RX claims:** Rybrevant NDCs; only patients **not** in the MX claims; `prescriber_npi` → account; amount = fills.
- **McKesson:**
  - **all Rybrevant alerts** (IV, FASPRO, Rybrevant + TKI); `physician_npi` → account; amount = alerts;
  - an H&N diagnosis (HNSCC / head / neck) marks H&N presence.
- **Chargemaster:**
  - **monthly** rows that overlap the 52 weeks, summed per NPI; NPI → account;
  - NSCLC encounters = Lung, HNSCC encounters = H&N;
  - no product column and no week.
- Prints, per source and product: rows, amount, and % placed on an account.

In [ ]:
in_window = lambda c: F.to_date(c).between(F.lit(START_DATE), F.lit(END_DATE))
parts = []

# DDD (all history kept: the 13-week look-back needs weeks before the window)
ddd = (df["ddd"].withColumn("product", ryb_product_name(F.col("prod_nm"))).filter(F.col("product").isin(LUNG_PRODUCTS))
                .select(F.col("jnj_id").cast("string").alias("jnj_id"), F.to_date("caln_dt").alias("day"),
                        week_of("caln_dt").alias("week"), "product", F.col("equiv_uns").cast("double").alias("amount")))
ddd = to_account(ddd, "jnj_id").cache()
parts.append(ddd.select("account_id", "day", "week", F.lit("DDD").alias("source"), "product", "amount", "jnj_id"))

# Xponent
xp = (df["xpon"].withColumn("product", ryb_product_name(F.col("prod_nm"))).filter(F.col("product").isin(LUNG_PRODUCTS))
                .select(clean(F.col("jnj_id")).alias("jnj_id"), F.to_date("caln_dt").alias("day"), week_of("caln_dt").alias("week"),
                        "product", F.col("equ_trx").cast("double").alias("amount")))
parts.append(to_account(xp, "jnj_id").select("account_id", "day", "week", F.lit("XPONENT").alias("source"), "product", "amount", "jnj_id"))

# MX claims
mx = (df["mx"].withColumn("product", ryb_product_ndc(F.col("ndc"))).filter(F.col("product").isin(LUNG_PRODUCTS))
              .select(F.concat_ws("|", "dv1", "dv2").alias("patient_id"), F.to_date("service_date").alias("day"),
                      "ndc", "product", clean(F.col("rendering_npi")).alias("npi")))
mx_patients = mx.select("patient_id").distinct()
mx_inf = (mx.dropDuplicates(["patient_id", "day", "ndc"]).join(npi_jnj, "npi", "left")
            .select("jnj_id", "day", week_of("day").alias("week"), "product", F.lit(1.0).alias("amount")))
parts.append(to_account(mx_inf, "jnj_id").select("account_id", "day", "week", F.lit("MX").alias("source"), "product", "amount", "jnj_id"))

# RX claims (patients not in MX)
rx = (df["rx"].withColumn("product", ryb_product_ndc(F.col("ndc"))).filter(F.col("product").isin(LUNG_PRODUCTS))
              .select(F.concat_ws("|", "dv1", "dv2").alias("patient_id"), "claim_id", F.to_date("fill_date").alias("day"),
                      "product", clean(F.col("prescriber_npi")).alias("npi"))
              .join(mx_patients, "patient_id", "left_anti").dropDuplicates(["claim_id"])
              .join(npi_jnj, "npi", "left")
              .select("jnj_id", "day", week_of("day").alias("week"), "product", F.lit(1.0).alias("amount")))
parts.append(to_account(rx, "jnj_id").select("account_id", "day", "week", F.lit("RX").alias("source"), "product", "amount", "jnj_id"))

# McKesson (all Rybrevant alerts)
an = F.upper(F.coalesce(F.col("alert_name"), F.lit("")))
mk_all = df["mckesson"].filter(an.contains("RYB") & in_window(F.col("diagnosis_date")))
dx_txt = F.upper(F.coalesce(F.col("diagnosis"), F.lit("")))
mk_hn = (mk_all.filter(dx_txt.contains("HNSCC") | dx_txt.contains("HEAD") | dx_txt.contains("NECK"))
               .select(clean(F.col("physician_npi")).alias("npi")).join(npi_jnj, "npi", "left"))
mk_hn = to_account(mk_hn, "jnj_id").filter("account_id IS NOT NULL").select("account_id").distinct()
mk = (mk_all.select(clean(F.col("physician_npi")).alias("npi"), F.to_date("diagnosis_date").alias("day"),
                    week_of("diagnosis_date").alias("week"),
                    F.when(an.contains("FASPRO"), "RYBREVANT FASPRO").when(an.contains("IV"), "RYBREVANT")
                     .otherwise("RYBREVANT (unspecified)").alias("product"), F.lit(1.0).alias("amount"))
            .join(npi_jnj, "npi", "left"))
parts.append(to_account(mk, "jnj_id").select("account_id", "day", "week", F.lit("MCKESSON").alias("source"), "product", "amount", "jnj_id"))

# Chargemaster: monthly rows overlapping the window, summed per NPI
desc = F.trim(pick(cm, "Time Frame Descriptive"))
p_start = F.to_date(F.regexp_extract(desc, r"(\d{4}-\d{2}-\d{2})\s*to", 1))
p_end = F.to_date(F.regexp_extract(desc, r"to\s*(\d{4}-\d{2}-\d{2})", 1))
cm_rows = (cm.filter(F.upper(pick(cm, "Cadence")).contains("MONTH"))
             .filter((p_end >= F.lit(START_DATE)) & (p_start <= F.lit(END_DATE)))
             .groupBy(clean(pick(cm, "NPI")).alias("npi"))
             .agg(F.sum(clean(pick(cm, "# EncountersNSCLC")).cast("double")).alias("nsclc"),
                  F.sum(clean(pick(cm, "# EncountersHNSCC")).cast("double")).alias("hnscc"))
             .join(npi_jnj, "npi", "left"))
cm_acc = to_account(cm_rows, "jnj_id").cache()
parts.append(cm_acc.select("account_id", F.lit(None).cast("date").alias("day"), F.lit(None).cast("date").alias("week"),
                           F.lit("CHARGEMASTER").alias("source"), F.lit("NO PRODUCT (Chargemaster)").alias("product"),
                           F.col("nsclc").alias("amount"), "jnj_id"))

activity = parts[0]
for p in parts[1:]:
    activity = activity.unionByName(p)
activity = activity.filter(F.coalesce("amount", F.lit(0.0)) > 0).cache()

in_win = F.col("day").isNull() | F.col("day").between(F.lit(START_DATE), F.lit(END_DATE))
(activity.filter(in_win).groupBy("source", "product")
         .agg(F.count("*").alias("rows"), F.round(F.sum("amount"), 1).alias("amount"),
              F.round(100 * F.sum(F.when(F.col("account_id").isNotNull(), F.col("amount")).otherwise(0)) / F.sum("amount"), 1)
               .alias("pct_amount_on_account"),
              F.min("week").alias("first_week"), F.max("week").alias("last_week"))
         .orderBy("source", "product").show(truncate=False))
print("Chargemaster HCPs with HNSCC encounters in window:", cm_acc.filter("hnscc > 0").select("npi").distinct().count())

## 6. Account segments (past 52 weeks)

### What this cell does: segment every L2/L3 account over the 52 weeks
- **Lung universe:** any Rybrevant activity (IV or FASPRO) at the account in the 52 weeks, in any source.
- **H&N universe:** LE.AI HCPs + Chargemaster HNSCC + McKesson H&N diagnosis.
- **Segment:** H&N only / H&N + Lung / Lung only.
- Prints:
  1. H&N universe accounts per source;
  2. segment counts by L2 / L3;
  3. activity per source, plus DDD split into IV and FASPRO units;
  4. **comparison with the FASPRO-only YTD definition:** how many accounts move between segments when IV and the 52 weeks are added.

In [ ]:
lung_acct = (activity.filter(in_win & F.col("account_id").isNotNull())
             .groupBy("account_id").pivot("source", ["DDD", "XPONENT", "MX", "RX", "MCKESSON", "CHARGEMASTER"])
             .agg(F.round(F.sum("amount"), 2)).fillna(0)
             .withColumn("lung_present", F.lit(True)))
ddd_prod = (activity.filter(in_win & F.col("account_id").isNotNull() & (F.col("source") == "DDD"))
            .groupBy("account_id").pivot("product", ["RYBREVANT", "RYBREVANT FASPRO"]).agg(F.round(F.sum("amount"), 2)).fillna(0)
            .withColumnRenamed("RYBREVANT", "DDD_IV").withColumnRenamed("RYBREVANT FASPRO", "DDD_FASPRO"))
lung_acct = lung_acct.join(ddd_prod, "account_id", "left").fillna(0, subset=["DDD_IV", "DDD_FASPRO"])

zero = lambda d: d.withColumn("hn_hcps", F.lit(0)).withColumn("hn_pts_2y", F.lit(0.0))
hn_set = hn_accounts.withColumn("hn_src", F.lit("LE.AI"))
if CM_HNSCC_AS_HN:
    hn_set = hn_set.unionByName(zero(cm_acc.filter("hnscc > 0 AND account_id IS NOT NULL").select("account_id").distinct())
                                .withColumn("hn_src", F.lit("CHARGEMASTER_HNSCC")))
if MCKESSON_HN_DX_AS_HN:
    hn_set = hn_set.unionByName(zero(mk_hn).withColumn("hn_src", F.lit("MCKESSON_HN_DX")))
print("H&N universe accounts by source:")
hn_set.groupBy("hn_src").agg(F.countDistinct("account_id").alias("accounts")).show(truncate=False)
hn_set = (hn_set.groupBy("account_id").agg(F.max("hn_hcps").alias("hn_hcps"), F.max("hn_pts_2y").alias("hn_pts_2y"),
                                            F.concat_ws(",", F.sort_array(F.collect_set("hn_src"))).alias("hn_sources"))
                .withColumn("hn_present", F.lit(True)).cache())

def segment_of(hn_col, lung_col):
    return (F.when(F.col(hn_col) & F.col(lung_col), "H&N + Lung").when(F.col(hn_col), "H&N only").otherwise("Lung only"))

seg_order = ["H&N only", "H&N + Lung", "Lung only"]
acct_seg = (hn_set.join(lung_acct, "account_id", "full").join(account_level, "account_id", "left")
            .fillna(False, subset=["hn_present", "lung_present"]).fillna(0)
            .withColumn("account_level", F.coalesce("account_level", F.lit("UNKNOWN")))
            .withColumn("segment", segment_of("hn_present", "lung_present"))
            .cache())

tbl = acct_seg.groupBy("account_level").pivot("segment", seg_order).count().fillna(0)
tbl = tbl.withColumn("Total", sum(F.col(f"`{x}`") for x in seg_order))
tbl.unionByName(tbl.agg(*[F.sum(F.col(f"`{x}`")).alias(x) for x in seg_order + ["Total"]]).withColumn("account_level", F.lit("ALL"))) \
   .orderBy("account_level").show(truncate=False)
(acct_seg.groupBy("segment")
         .agg(F.count("*").alias("accounts"), F.sum("hn_hcps").alias("hn_hcps"),
              *[F.round(F.sum(x), 1).alias(x) for x in ["DDD", "DDD_IV", "DDD_FASPRO", "XPONENT", "MX", "RX", "MCKESSON", "CHARGEMASTER"]])
         .withColumn("pct_accounts", F.round(100 * F.col("accounts") / F.sum("accounts").over(Window.partitionBy()), 1))
         .orderBy("segment").show(truncate=False))

# comparison: FASPRO-only YTD 2026 Lung definition (previous notebook) vs this one
old_lung = (activity.filter(F.col("account_id").isNotNull() &
                            (((F.col("product") == "RYBREVANT FASPRO") & (F.col("day") >= F.lit("2026-01-01")) & (F.col("day") <= F.lit(END_DATE)))
                             | (F.col("source") == "CHARGEMASTER")))
                    .select("account_id").distinct().withColumn("lung_old", F.lit(True)))
cmp = (acct_seg.select("account_id", "hn_present", "segment").join(old_lung, "account_id", "full")
               .fillna(False, subset=["hn_present", "lung_old"])
               .withColumn("segment", F.coalesce("segment", F.lit("not in table")))
               .withColumn("segment_faspro_ytd", segment_of("hn_present", "lung_old")))
print("Accounts: FASPRO-only YTD segment (rows) vs Rybrevant IV + FASPRO 52 weeks (columns)")
cmp.groupBy("segment_faspro_ytd").pivot("segment", seg_order).count().fillna(0).orderBy("segment_faspro_ytd").show(truncate=False)

## 7. Weekly DDD split

### What this cell does: put each week's DDD units (`SPLIT_PRODUCTS`) into a segment
- **DDD rows:** products in `SPLIT_PRODUCTS` (default FASPRO), weeks starting in the window, summed per account × week.
- **Lung present that week:**
  - the account is in the Lung universe from the **non-DDD** sources (52 weeks), **or**
  - it bought **any Rybrevant (IV or FASPRO) in DDD** in the 13 weeks **before** that week.
- **H&N present:** from section 6.
- **Segment:** H&N + Lung / H&N only / Lung only / Unclassified (no signal) / Unmapped account (DDD buyer not in OPAL).
- **Outputs:**
  1. units per segment;
  2. units per week per segment;
  3. latest weeks per account;
  4. latest week by level.

In [ ]:
ddd_aw = (ddd.filter(F.col("product").isin(SPLIT_PRODUCTS) & F.col("week").between(F.lit(START_DATE), F.lit(END_DATE)))
             .groupBy("account_id", "week").agg(F.round(F.sum("amount"), 3).alias("ddd_units")))

ddd_hist = (ddd.filter("account_id IS NOT NULL").select(F.col("account_id").alias("a_acc"), F.col("week").alias("a_week")).distinct())
look = (ddd_aw.filter("account_id IS NOT NULL")
        .join(ddd_hist, (F.col("account_id") == F.col("a_acc")) &
                        (F.col("a_week") >= F.date_sub("week", LOOKBACK_WEEKS * 7)) & (F.col("a_week") <= F.date_sub("week", 7)), "left")
        .groupBy("account_id", "week").agg(F.count("a_week").alias("ddd_weeks_prior_13w")))
hist_lung = (activity.filter((F.col("source") != "DDD") & F.col("account_id").isNotNull() & in_win)
                     .groupBy("account_id").agg(F.concat_ws(",", F.sort_array(F.collect_set("source"))).alias("hist_lung_sources")))

weekly = (ddd_aw.join(look, ["account_id", "week"], "left").join(hist_lung, "account_id", "left")
                .join(hn_set.select("account_id", "hn_present", "hn_hcps", "hn_sources"), "account_id", "left")
                .join(account_level, "account_id", "left")
                .fillna(False, subset=["hn_present"]).fillna(0, subset=["hn_hcps", "ddd_weeks_prior_13w"])
                .withColumn("lung_present", F.col("hist_lung_sources").isNotNull() | (F.col("ddd_weeks_prior_13w") > 0))
                .withColumn("segment", F.when(F.col("account_id").isNull(), "Unmapped account")
                                        .when(F.col("hn_present") & F.col("lung_present"), "H&N + Lung")
                                        .when(F.col("hn_present"), "H&N only")
                                        .when(F.col("lung_present"), "Lung only")
                                        .otherwise("Unclassified"))
                .cache())

all_segs = seg_order + ["Unclassified", "Unmapped account"]
print("Products split:", SPLIT_PRODUCTS)
print("1) DDD units per segment (whole window)")
(weekly.groupBy("segment").agg(F.round(F.sum("ddd_units"), 1).alias("ddd_units"),
                               F.countDistinct("account_id").alias("accounts"), F.count("*").alias("account_weeks"))
       .withColumn("pct_units", F.round(100 * F.col("ddd_units") / F.sum("ddd_units").over(Window.partitionBy()), 1))
       .orderBy("segment").show(truncate=False))

print("2) DDD units per week per segment")
weekly.groupBy("week").pivot("segment", all_segs).agg(F.round(F.sum("ddd_units"), 1)).fillna(0).orderBy("week").show(60, False)

last_weeks = [r["week"] for r in weekly.select("week").distinct().orderBy(F.desc("week")).limit(RECENT_WEEKS).collect()]
print(f"3) Latest {RECENT_WEEKS} weeks, per account:", [str(w) for w in sorted(last_weeks)])
(weekly.filter(F.col("week").isin(last_weeks))
       .select("week", "account_id", "account_level", "ddd_units", "segment", "hn_sources", "hist_lung_sources", "ddd_weeks_prior_13w")
       .orderBy(F.desc("week"), F.desc("ddd_units")).show(50, False))

latest = max(last_weeks) if last_weeks else None
print("4) Latest week", latest, "- accounts per segment by level")
(weekly.filter(F.col("week") == F.lit(latest)).groupBy(F.coalesce("account_level", F.lit("UNMAPPED")).alias("account_level"))
       .pivot("segment", all_segs).count().fillna(0).orderBy("account_level").show(truncate=False))

## 8. Account segment insights (L2/L3)

### What this cell does: insight table + scorecard
- One row per account (`ins`):
  - name (from LE.AI where available);
  - number of Lung sources;
  - weak-Lung flag (Chargemaster only);
  - **IV → FASPRO transition status** from DDD: IV only / FASPRO only / both.
- **Scorecard by level × segment:**
  - accounts;
  - DDD units (total, IV, FASPRO) and % of units;
  - units per account;
  - H&N doctors and H&N treating patients;
  - average number of Lung sources.
- **Transition table:** segment × IV / FASPRO buying pattern.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)

SEG_COLORS = {"H&N only": "#4C78A8", "H&N + Lung": "#B279A2", "Lung only": "#54A24B",
              "Unclassified": "#BAB0AC", "Unmapped account": "#E45756"}
LUNG_SRC = ["DDD", "XPONENT", "MX", "RX", "MCKESSON", "CHARGEMASTER"]

acct_names = (coh.filter("cohort_account_id IS NOT NULL")
                 .select(F.col("cohort_account_id").alias("account_id"), F.col("cohort_account_name").alias("account_name"))
                 .dropDuplicates(["account_id"]))
ins = (acct_seg.join(acct_names, "account_id", "left")
       .withColumn("n_lung_sources", sum((F.col(x) > 0).cast("int") for x in LUNG_SRC))
       .withColumn("weak_lung_chargemaster_only", (F.col("n_lung_sources") == 1) & (F.col("CHARGEMASTER") > 0))
       .withColumn("ddd_product_mix", F.when((F.col("DDD_IV") > 0) & (F.col("DDD_FASPRO") > 0), "IV + FASPRO")
                                       .when(F.col("DDD_IV") > 0, "IV only").when(F.col("DDD_FASPRO") > 0, "FASPRO only")
                                       .otherwise("no DDD"))
       .cache())

score = (ins.groupBy("account_level", "segment")
            .agg(F.count("*").alias("accounts"), F.round(F.sum("DDD"), 1).alias("ddd_units"),
                 F.round(F.sum("DDD_IV"), 1).alias("ddd_iv"), F.round(F.sum("DDD_FASPRO"), 1).alias("ddd_faspro"),
                 F.sum("hn_hcps").alias("hn_hcps"), F.round(F.sum("hn_pts_2y"), 0).alias("hn_treating_pts_2y"),
                 F.round(F.avg("n_lung_sources"), 2).alias("avg_lung_sources"))
            .toPandas())
score["pct_accounts"] = (100 * score.accounts / score.accounts.sum()).round(1)
score["pct_ddd_units"] = (100 * score.ddd_units / max(score.ddd_units.sum(), 1e-9)).round(1)
score["units_per_account"] = (score.ddd_units / score.accounts).round(2)
print(score.sort_values(["account_level", "segment"]).to_string(index=False))

print("\nIV -> FASPRO transition: accounts by DDD product mix")
ins.groupBy("ddd_product_mix").pivot("segment", seg_order).count().fillna(0).orderBy("ddd_product_mix").show(truncate=False)

### What this cell does: figures
1. Accounts and DDD units (IV + FASPRO) per segment, L2 vs L3.
2. Weekly DDD units by product (IV vs FASPRO): the transition.
3. Weekly DDD units (`SPLIT_PRODUCTS`) by segment, from section 7.
4. Concentration of DDD units across accounts.
5. H&N + Lung accounts: DDD units vs H&N treating patients.

In [ ]:
segs = ["H&N only", "H&N + Lung", "Lung only"]
lv = score.pivot_table(index="segment", columns="account_level", values=["accounts", "ddd_units"], aggfunc="sum").fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric, title in [(axes[0], "accounts", "Accounts per segment"), (axes[1], "ddd_units", "DDD units (IV + FASPRO), 52 weeks")]:
    t = lv[metric].reindex(segs).fillna(0)
    t.plot(kind="bar", stacked=True, ax=ax, color=["#1f77b4", "#ff7f0e", "#999999"][:t.shape[1]])
    ax.set_title(title); ax.set_xlabel(""); plt.setp(ax.get_xticklabels(), rotation=0)
    for i, v in enumerate(t.sum(axis=1)):
        ax.text(i, v, f"{v:,.0f}", ha="center", va="bottom")
plt.tight_layout(); plt.show()

tr = (ddd.filter(F.col("week").between(F.lit(START_DATE), F.lit(END_DATE)))
         .groupBy("week").pivot("product", ["RYBREVANT", "RYBREVANT FASPRO"]).agg(F.sum("amount")).fillna(0)
         .orderBy("week").toPandas().set_index("week"))
fig, ax = plt.subplots(figsize=(13, 4))
for col, c in [("RYBREVANT", "#F58518"), ("RYBREVANT FASPRO", "#4C78A8")]:
    if col in tr.columns:
        ax.plot(tr.index, tr[col], label=col.replace("RYBREVANT", "Rybrevant IV") if col == "RYBREVANT" else "Rybrevant FASPRO",
                color=c, linewidth=2)
ax.set_title("Weekly DDD units: Rybrevant IV vs FASPRO (all buyers)"); ax.set_ylabel("equivalized units"); ax.legend()
plt.tight_layout(); plt.show()

wk = (weekly.groupBy("week").pivot("segment", list(SEG_COLORS)).agg(F.sum("ddd_units")).fillna(0)
            .orderBy("week").toPandas().set_index("week"))
fig, ax = plt.subplots(figsize=(13, 4))
for col in wk.columns:
    ax.plot(wk.index, wk[col], label=col, color=SEG_COLORS.get(col), linewidth=2)
ax.set_title("Weekly DDD units by segment (" + ", ".join(SPLIT_PRODUCTS) + ")"); ax.set_ylabel("equivalized units")
ax.legend(ncol=5, fontsize=8)
plt.tight_layout(); plt.show()

p = ins.filter("DDD > 0").select("account_id", "DDD").toPandas().sort_values("DDD", ascending=False)
p["cum_pct"] = 100 * p.DDD.cumsum() / p.DDD.sum()
for n in (10, 50, 100):
    if len(p) >= n:
        print(f"Top {n} accounts buy {p.cum_pct.iloc[n - 1]:.1f}% of mapped DDD units (of {len(p):,} buying accounts)")
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(p) + 1), p.cum_pct.values); ax.axhline(80, ls="--", color="grey")
ax.set_xlabel("accounts (ranked by DDD units)"); ax.set_ylabel("cumulative % of units"); ax.set_title("Concentration of DDD")
plt.tight_layout(); plt.show()

sc = ins.filter("segment = 'H&N + Lung'").select("DDD", "hn_pts_2y", "account_level").toPandas()
fig, ax = plt.subplots(figsize=(6, 4))
for lvl, c in [("L2", "#1f77b4"), ("L3", "#ff7f0e")]:
    d = sc[sc.account_level == lvl]
    ax.scatter(d.DDD, d.hn_pts_2y, s=14, alpha=0.6, label=lvl, color=c)
ax.set_xlabel("DDD units (IV + FASPRO), 52 weeks"); ax.set_ylabel("H&N treating patients (LE.AI, 2y)")
ax.set_title("H&N + Lung accounts"); ax.legend()
plt.tight_layout(); plt.show()

### What this cell does: actionable account lists
1. **H&N only:** H&N HCPs, no Rybrevant use, ranked by H&N treating patients.
2. **H&N + Lung:** ranked by DDD units, with IV / FASPRO split.
3. **Lung only:** largest buyers without an H&N signal.
4. **Signal strength:** accounts by number of Lung sources; Chargemaster-only flagged.

In [ ]:
cols = ["account_id", "account_name", "account_level", "hn_hcps", "hn_pts_2y", "DDD", "DDD_IV", "DDD_FASPRO", "XPONENT", "MX",
        "n_lung_sources", "hn_sources"]
print("1) H&N only - no Rybrevant use yet")
ins.filter("segment = 'H&N only'").orderBy(F.desc("hn_pts_2y"), F.desc("hn_hcps")).select(*cols).show(20, False)
print("2) H&N + Lung - largest Rybrevant buyers with H&N exposure")
ins.filter("segment = 'H&N + Lung'").orderBy(F.desc("DDD")).select(*cols).show(20, False)
print("3) Lung only - largest Rybrevant buyers without an H&N signal")
ins.filter("segment = 'Lung only'").orderBy(F.desc("DDD")).select(*cols).show(20, False)
print("4) Signal strength - accounts by number of Lung sources")
(ins.filter("segment != 'H&N only'").groupBy("segment").pivot("n_lung_sources").count().fillna(0).orderBy("segment").show())
print("Accounts whose only Lung signal is Chargemaster (no product column):", ins.filter("weak_lung_chargemaster_only").count())

## 9. Segment profiles

### What this cell does: describe who is in each segment
1. **Specialty** of H&N doctors and Rybrevant writers (dmn_cust `prim_spclt_nm`).
2. **Doctors per account** and **dual doctors**: on the LE.AI H&N list **and** writing Rybrevant at the same account.
3. **Account type** (dmn_cust `cust_typ_nm`).
4. **Practice setting** (LE.AI, H&N accounts).
5. **Where Rybrevant is given** (MX): infusions, patients, facility lines, 340B lines, IV vs FASPRO.
6. **Adoption** (DDD): first purchase, weeks active, last 13 weeks vs previous 13.

In [ ]:
seg_of = ins.select("account_id", "segment")
win = F.col("day").between(F.lit(START_DATE), F.lit(END_DATE))
dmn_info = (df["dmn"].select(clean(F.col("jnj_id")).alias("jnj_id"), F.col("prim_spclt_nm").alias("specialty"),
                             F.col("cust_typ_nm").alias("cust_type"))
                     .filter("jnj_id IS NOT NULL").dropDuplicates(["jnj_id"]).cache())

hn_hcp = (coh.withColumn("account_id", acc_col).filter("account_id IS NOT NULL AND jnj_id IS NOT NULL")
             .select("account_id", "jnj_id", "npi").distinct())
writers = None
for d in (mx_inf, rx, xp.filter("amount > 0"), mk):
    t = to_account(d.filter(win).select("jnj_id"), "jnj_id").filter("account_id IS NOT NULL AND jnj_id IS NOT NULL")
    writers = t if writers is None else writers.unionByName(t)
writers = writers.distinct()
hcps = (hn_hcp.select("account_id", "jnj_id").withColumn("role", F.lit("H&N HCP"))
        .unionByName(writers.withColumn("role", F.lit("Rybrevant writer")))
        .join(seg_of, "account_id").join(dmn_info.select("jnj_id", "specialty"), "jnj_id", "left").cache())

sp = (hcps.groupBy("segment", "role", F.coalesce("specialty", F.lit("UNKNOWN")).alias("specialty"))
          .agg(F.countDistinct("jnj_id").alias("hcps")).toPandas())
print("1) Specialty mix")
for (sg, rl), g in sp.groupby(["segment", "role"]):
    g = g.sort_values("hcps", ascending=False)
    g["pct"] = (100 * g.hcps / g.hcps.sum()).round(1)
    print(f"\n{sg} | {rl} | {int(g.hcps.sum()):,} HCPs")
    print(g.head(8)[["specialty", "hcps", "pct"]].to_string(index=False))

dual = (hn_hcp.select("account_id", "jnj_id").join(writers, ["account_id", "jnj_id"])
              .groupBy("account_id").agg(F.countDistinct("jnj_id").alias("dual_hcps")))
nw = writers.groupBy("account_id").agg(F.countDistinct("jnj_id").alias("ryb_writers"))
per = ins.select("account_id", "segment", "hn_hcps").join(nw, "account_id", "left").join(dual, "account_id", "left").fillna(0)
print("\n2) Doctors per account")
(per.groupBy("segment").agg(F.count("*").alias("accounts"), F.round(F.avg("hn_hcps"), 1).alias("avg_hn_hcps"),
                            F.round(F.avg("ryb_writers"), 1).alias("avg_ryb_writers"),
                            F.sum("dual_hcps").alias("dual_hcps_total"),
                            F.sum((F.col("dual_hcps") > 0).cast("int")).alias("accounts_with_dual_hcp"))
    .orderBy("segment").show(truncate=False))

print("3) Account type (dmn_cust)")
(ins.select("account_id", "segment").join(dmn_info.select(F.col("jnj_id").alias("account_id"), "cust_type"), "account_id", "left")
    .groupBy(F.coalesce("cust_type", F.lit("NOT IN DMN_CUST")).alias("account_type"))
    .pivot("segment", seg_order).count().fillna(0).orderBy(F.desc("`H&N + Lung`")).show(20, False))

ps = h.select(clean(pick(h, "NPI")).alias("npi"), clean(pick(h, "Practice Settings")).alias("practice_setting")).dropDuplicates(["npi"])
acc_ps = (hn_hcp.join(ps, "npi").filter("practice_setting IS NOT NULL").groupBy("account_id", "practice_setting").count()
                .withColumn("rk", F.row_number().over(Window.partitionBy("account_id").orderBy(F.desc("count"), "practice_setting")))
                .filter("rk = 1").select("account_id", "practice_setting"))
print("4) Practice setting (LE.AI, H&N accounts)")
(ins.filter("segment != 'Lung only'").select("account_id", "segment").join(acc_ps, "account_id", "left")
    .groupBy(F.coalesce("practice_setting", F.lit("UNKNOWN")).alias("practice_setting"))
    .pivot("segment", ["H&N only", "H&N + Lung"]).count().fillna(0).orderBy(F.desc("`H&N only`")).show(20, False))

mods = F.coalesce(F.col("modifiers"), F.lit(""))
mxs = (df["mx"].withColumn("product", ryb_product_ndc(F.col("ndc"))).filter(F.col("product").isin(LUNG_PRODUCTS))
        .select(F.concat_ws("|", "dv1", "dv2").alias("patient_id"), F.to_date("service_date").alias("day"), "ndc", "product",
                clean(F.col("rendering_npi")).alias("npi"), F.upper("data_source").alias("data_source"),
                (mods.contains("|TB|") | mods.contains("|UD|")).cast("int").alias("is_340b"))
        .filter(win).join(npi_jnj, "npi", "left"))
mxs = to_account(mxs, "jnj_id").join(seg_of, "account_id")
print("5) Rybrevant infusions by care setting and product (MX)")
(mxs.groupBy("segment", "product")
    .agg(F.countDistinct(F.concat_ws("|", "patient_id", "day", "ndc")).alias("infusions"),
         F.countDistinct("patient_id").alias("patients"),
         F.round(100 * F.avg((F.col("data_source") == "INSTITUTIONAL").cast("int")), 1).alias("pct_facility_lines"),
         F.round(100 * F.avg("is_340b"), 1).alias("pct_340b_lines"))
    .orderBy("segment", "product").show(truncate=False))

last_wk = ddd.agg(F.max("week")).collect()[0][0]
d13 = F.date_sub(F.lit(last_wk), 13 * 7); d26 = F.date_sub(F.lit(last_wk), 26 * 7)
ad = (ddd.filter(F.col("account_id").isNotNull() & F.col("week").between(F.lit(START_DATE), F.lit(END_DATE))).groupBy("account_id")
         .agg(F.min("week").alias("first_week"), F.countDistinct("week").alias("weeks_active"),
              F.sum(F.when(F.col("week") > d13, F.col("amount")).otherwise(0)).alias("last13"),
              F.sum(F.when((F.col("week") > d26) & (F.col("week") <= d13), F.col("amount")).otherwise(0)).alias("prev13"))
         .join(seg_of, "account_id"))
print("6) Adoption (DDD IV + FASPRO buyers), latest week:", last_wk)
(ad.groupBy("segment")
   .agg(F.count("*").alias("buying_accounts"), F.min("first_week").alias("earliest_buy"),
        F.round(F.avg("weeks_active"), 1).alias("avg_weeks_active"),
        F.sum((F.col("first_week") > d13).cast("int")).alias("new_last_13w"),
        F.sum((F.col("last13") > F.col("prev13")).cast("int")).alias("growing"),
        F.sum((F.col("last13") < F.col("prev13")).cast("int")).alias("declining"),
        F.sum((F.col("last13") == 0).cast("int")).alias("stopped_last_13w"))
   .orderBy("segment").show(truncate=False))

## 10. Save to the Flow

### What this cell does: create the output datasets (if missing) and write the results
- Set `SAVE = True` to write.
- Missing datasets are created in the Flow automatically, on the same connection as `RYB_DDD_Sales_copy`.
- Outputs:
  - `HN_RYB_ACCOUNT_SEGMENTS_52W`: one row per L2/L3 account with its segment, name, H&N volumes, Lung sources, DDD IV / FASPRO units.
  - `HN_RYB_DDD_WEEKLY_SEGMENTS_52W`: one row per account × week with DDD units (`SPLIT_PRODUCTS`) and that week's segment.

In [ ]:
SAVE = False
outputs = {"HN_RYB_ACCOUNT_SEGMENTS_52W": ins, "HN_RYB_DDD_WEEKLY_SEGMENTS_52W": weekly}
if SAVE:
    client = dataiku.api_client()
    project = client.get_default_project()
    conn = project.get_dataset(DATASETS["ddd"]).get_settings().get_raw()["params"]["connection"]
    existing = [d["name"] for d in project.list_datasets()]
    for name, d in outputs.items():
        if name not in existing:
            b = project.new_managed_dataset(name)
            b.with_store_into(conn)
            b.create()
            print("created", name)
        dkuspark.write_with_schema(dataiku.Dataset(name), d)
        print("saved", name)